****MODEL EVALUATION****

*Load Libraries and Prediction Values*

In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

In [3]:
from pathlib import Path

artifact_dir = Path(".")

logistic_model = joblib.load(artifact_dir / "logistic_regression_model.pkl")

x_test = joblib.load(artifact_dir / "X_test_processed.pkl")

y_test = joblib.load(artifact_dir / "y_test.pkl")

print("Model and test data loaded successfully.")
print("Test observations:", len(y_test))

Model and test data loaded successfully.
Test observations: 20000


In [4]:
y_pred = logistic_model.predict(x_test)

y_prob = logistic_model.predict_proba(x_test)[:, 1]

print("Predicted classes:", y_pred[:10])
print("Fraud probabilities:", y_prob[:10])

Predicted classes: [0 0 0 0 0 0 0 0 0 0]
Fraud probabilities: [0.00736796 0.00835705 0.00610112 0.01113796 0.00653791 0.00892048
 0.01039081 0.00737931 0.01005557 0.0078273 ]


*Evaluate the Classifier*

In [5]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC (Average Precision):", average_precision_score(y_test, y_prob))

Accuracy: 0.99
Precision: 0.0
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.47641287878787875
PR-AUC (Average Precision): 0.009926480400578802


## Interpretation of the Evaluation Results

The model has an **accuracy of 0.99**, meaning 99% of the test transactions were classified correctly. However, accuracy is misleading for this imbalanced dataset: fraud represents only about **1%** of the test set. The model predicted no transactions as fraud, so it missed all **200 fraud cases**. Correctly classifying the many legitimate transactions produced a high accuracy despite failing to detect fraud.

**Precision = 0.00** means the model made no correct fraud predictions. Since it predicted no transactions as fraud, precision is mathematically undefined; the metric reports 0 because `zero_division=0` was specified. **Recall = 0.00** means it detected none of the fraud cases. Consequently, the **F1-score = 0.00**, since F1 combines precision and recall and both are zero.

**ROC-AUC = 0.4764** measures how well the model ranks fraud cases above legitimate cases across decision thresholds. A score of 0.5 represents chance-level ranking, so 0.4764 indicates below-chance ranking on this test set. Confirm that fraud is encoded as the positive class and that `y_prob` contains the fraud probabilities; if both are correct, the model shows poor ability to distinguish the classes.

**PR-AUC (Average Precision) = 0.0099** summarizes the precision-recall trade-off, focusing on fraud as the positive class. Average Precision should be interpreted against the fraud prevalence, which is about **0.0100 (1%)** here. The score is slightly below that baseline, indicating very weak retrieval of fraud cases. The notebook reports Average Precision as its PR-AUC summary; it is related to, but not always identical to, a trapezoidally integrated area under the precision-recall curve.

**Overall deduction:** Despite its 99% accuracy, this model is not effective for fraud detection at the current decision threshold: it flags no fraud, and both ranking metrics indicate little useful separation. Accuracy alone should not be used to judge this model.